# LaneTrace quickstart: autolog, middleware and the fluent API

LaneTrace records an agent run as a sequence diagram: User → Orchestrator → LLM,
tools and sub-agents. This notebook traces one small LangChain agent three ways:

1. `lanetrace.langchain.autolog()`, without touching the agent code (like
   `mlflow.langchain.autolog()`);
2. `LaneTraceMiddleware`, attached to the agent;
3. the fluent API (`start_run`, `log_event`, `@lanetrace.trace`), for code that
   isn't LangChain.

It runs offline: a scripted chat model stands in for the LLM, traces go to a
SQLite file in a temporary directory, and the notebook reads them back through
the dashboard's API, served in-process. No server to start, no network, no API
key.

```bash
pip install 'lanetrace[langchain,server]' jupyter
```

## Setup

By default, traces go to `./lanetrace.db`, the file `lanetrace ui` reads when
started in the same directory. Here they go to a temporary directory instead.

In [ ]:
import tempfile
from pathlib import Path

import lanetrace

workdir = Path(tempfile.mkdtemp(prefix="lanetrace-notebook-"))
DB_FILE = workdir / "lanetrace.db"
lanetrace.set_tracking_uri(f"sqlite:///{DB_FILE}")   # like mlflow.set_tracking_uri()
lanetrace.set_project("notebook-quickstart")   # like mlflow.set_experiment()

The agent: a scripted chat model, which replays canned replies (and ignores the
tools `create_agent` binds to it), and one tool. The agent calls `add`, then
answers.

In [ ]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool


class ScriptedChatModel(FakeMessagesListChatModel):
    model_name: str = "scripted-llm"  # the LLM's lane in the diagram

    def bind_tools(self, tools, **kwargs):
        return self


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


def build_agent(**kwargs):
    usage = {"input_tokens": 42, "output_tokens": 9, "total_tokens": 51}
    model = ScriptedChatModel(
        responses=[
            AIMessage(content="", tool_calls=[{"name": "add", "args": {"a": 2, "b": 3}, "id": "call_1"}], usage_metadata=usage),
            AIMessage(content="2 + 3 = 5.", usage_metadata=usage),
        ]
    )
    return create_agent(model, tools=[add], system_prompt="You are a careful calculator.", name="calculator", **kwargs)


QUESTION = {"messages": [{"role": "user", "content": "What is 2 + 3?"}]}

To read the runs back, this helper asks the dashboard's own REST API
(`GET /api/runs/recent`, then `GET /api/runs/<run id>`), served in-process on
the same database: no port, no network. It prints the latest run, one line per
event, that is per arrow of the diagram.

In [ ]:
import json
import os

# What `lanetrace ui --database-url ... --auth none` sets up. The dashboard
# app reads DATABASE_URL when it is imported: restart the kernel to start over.
os.environ["DATABASE_URL"] = f"sqlite:///{DB_FILE}"
os.environ["LANETRACE_AUTH"] = "none"

from fastapi.testclient import TestClient

from lanetrace.server.app import app   # the app `lanetrace ui` serves

dashboard = TestClient(app, base_url="http://localhost")


def show_last_run(payloads=False):
    latest = dashboard.get("/api/runs/recent", params={"limit": 1}).json()["runs"][0]
    run = dashboard.get(f"/api/runs/{latest['id']}").json()["run"]
    print(f"run {run['name']!r} in project {run['project']['name']!r}: {run['status']}")
    for event in run["events"]:
        print(f"  {event['type']:<12} {event['source']:>14} → {event['target']:<14} {event['label'] or ''}")
        if payloads:
            print(f"  {'':12} payload {json.dumps(event['payload'], ensure_ascii=False)[:110]}")

## 1. `autolog()`: no change to the agent

`autolog()` registers a LangChain callback for the whole process. Each top-level
invocation becomes one run, named after the agent, in the current project. LLM
calls carry an output preview and the token usage. `autolog(disable=True)` turns
it off.

In [ ]:
import lanetrace.langchain

lanetrace.langchain.autolog()
result = build_agent().invoke(QUESTION)
print(result["messages"][-1].content)
show_last_run(payloads=True)

## 2. `LaneTraceMiddleware`: attached to one agent

One middleware instance is one run, so use it when the agent is built for each
invocation. It also records the input of each LLM call (system prompt and
messages). Turn autolog off first, or this run would be traced twice.

In [ ]:
from lanetrace.langchain import LaneTraceMiddleware

lanetrace.langchain.autolog(disable=True)
agent = build_agent(middleware=[LaneTraceMiddleware(run_name="calculator (middleware)")])
agent.invoke(QUESTION)
show_last_run(payloads=True)

The middleware writes to the tracking URI too; given `url=` or `api_key=`, it
posts to that server instead. A server that compiles its agent once and reuses
it would get one run for all its requests: it needs a callback handler per
request instead, see `02_async_callback_handler.ipynb`.

## 3. The fluent API

`start_run()` opens a run (a context manager, like `mlflow.start_run()`),
`log_event()` adds an arrow, and `@lanetrace.trace` records a function call as a
`tool_call` followed by its `tool_result` (or an `error`). Event types are
`llm_call`, `tool_call`, `tool_result`, `handoff`, `error` and `final_answer`.

In [ ]:
@lanetrace.trace
def get_weather(city: str) -> str:
    return f"Sunny in {city}"


with lanetrace.start_run("weather (manual)"):
    lanetrace.log_event(source="User", target="Orchestrator", type="handoff", label="Weather in Paris?")
    lanetrace.log_event(source="Orchestrator", target="LLM", type="llm_call", label="plan", duration_ms=420)
    get_weather("Paris")
    lanetrace.log_event(source="Orchestrator", target="User", type="final_answer", label="It's sunny in Paris.")
show_last_run()

Inside `start_run()`, autologged invocations join that run instead of opening
their own, so one run can hold several agent calls:

In [ ]:
lanetrace.langchain.autolog()
with lanetrace.start_run("two questions, one run"):
    build_agent().invoke(QUESTION)
    build_agent().invoke({"messages": [{"role": "user", "content": "What is 2 + 3, again?"}]})
lanetrace.langchain.autolog(disable=True)
show_last_run()

## Open the runs in the dashboard

`lanetrace ui` serves the dashboard on http://127.0.0.1:3000, without a login
on localhost. Point it at this notebook's database:

In [ ]:
print(f"lanetrace ui --database-url sqlite:///{DB_FILE}")

## In your own code

Without `set_tracking_uri()`, the `LANETRACE_TRACKING_URI` environment variable
decides where traces go:

| Tracking URI | Traces go to |
| --- | --- |
| unset | `sqlite:///./lanetrace.db`, read by `lanetrace ui` started in the same directory |
| a SQLAlchemy URL (`sqlite:///...`, `postgresql://...`) | that database, written directly |
| `http://host:port` | a running `lanetrace ui`; set `LANETRACE_KEY` (a project key, `atr_...`) if it runs with `--auth password` |

When only the pre-0.2 `LANETRACE_URL` or `LANETRACE_KEY` is set, traces go to
that server (`http://localhost:3000` for a key alone). `LANETRACE_PROJECT` sets
the project when `set_project()` isn't called (default: `Default`).

Tracing never breaks the agent: events are written from a background thread, and
the first storage error turns tracing off for that run, with one logged warning.